# Model Selection and Adaptation Lab

Train and compare a small text classifier using governed splits, task slices, cost estimates, and a release decision.

Run the lecture cells in sequence with Python 3.10 or later. The default exercises use bundled data and require no model account.


## Practice case · A campus queue nobody wants to call twice

Hassan is piloting a routing classifier for campus support. Some students submit one short sentence from a phone. The help desk wants to resolve requests faster and wants difficult cases to reach a person instead of being routed confidently to the wrong queue.

The workplace scenario is fictional.


## Prepare data and learn a baseline


### Lecture 01 · Group-aware splits and dataset lineage

Split records by group and document lineage before training.

#### Put related incidents in the same split

Several students use the same residence template for a network issue. Randomly splitting individual tickets leaves near copies in training and test.

Group by site and incident pattern where possible. Keep related records together and report those grouping decisions with the split. A realistic holdout includes unfamiliar phrasing.

#### How it works

Allocate related examples to the same split. Grouping by source, customer, or template prevents paraphrases of one event appearing in both the train and holdout sets. Keep a version and record counts for each slice. A tiny fixture demonstrates the mechanics while leaving uncertainty visible.

##### Split independent work before fitting

Repeated templates and near-duplicate incidents can make random row splits misleading. If similar tickets appear in both training and evaluation, a model can learn a wording shortcut and still look as if it generalized. Choose a group such as site, original incident, or source document so related rows stay together.

Record fields as they existed at the moment the prediction is made. A status added by the support team after resolution is part of the outcome, even if it is present in a table you can train on. Excluding it may lower measured accuracy and make that evidence more representative of the deployed task.

Use training data to fit, validation data to choose settings, and a held-out test to report the final comparison. Record group membership and preprocessing versions. Review a few examples from each split to check that the separation is meaningful, not merely an artifact of the grouping key.

#### Worked example

Two requests copied from one campus support template share a group. Putting one copy in validation would make scores artificially optimistic.

#### Try it

Compute the group intersections across train, validation and test. Change one fixture so a group leaks across the split.


In [ ]:
from collections import Counter
from dataclasses import dataclass


@dataclass(frozen=True)
class Record:
    group: str
    slice: str
    text: str
    label: str


splits = {
    "train": (
        Record("a", "short", "wifi network disconnect", "network"),
        Record("a", "long", "wifi access point network slow", "network"),
        Record("b", "short", "router internet wifi", "network"),
        Record("c", "short", "projector screen dark", "display"),
        Record("d", "long", "screen projector flickers with cable", "display"),
    ),
    "validation": (
        Record("e", "short", "network wifi slow", "network"),
        Record("f", "short", "projector cable screen", "display"),
        Record("g", "long", "wifi classroom screen setup check", "display"),
    ),
    "test": (
        Record("h", "short", "router wifi down", "network"),
        Record("i", "long", "classroom projector screen dark today", "display"),
        Record("j", "short", "cable issue", "display"),
        Record("k", "long", "classroom needs attention before class", "network"),
    ),
}
dataset_version = "classroom-support-fixtures-v1"


def check_splits(partitions):
    owners = {}
    for name, records in partitions.items():
        for row in records:
            previous = owners.setdefault(row.group, name)
            assert previous == name, f"group {row.group} crosses {previous} and {name}"


check_splits(splits)
print({name: dict(Counter(row.slice for row in records)) for name, records in splits.items()})


**Compare your result**

The leakage assertion should fail and identify the crossing group before fitting begins.


### Lecture 02 · Fit and compare a supervised baseline

Train a Naive Bayes text classifier and compare its decisions with a majority-class baseline.

#### Make the baseline live through the same workload

The trained classifier beats the majority rule on examples written by staff. Its lead is uncertain on short student messages.

Fit both choices on the same training scope and compare on the same validation cases. Look at errors by source and length. A gain on polished examples may be irrelevant to workload routing.

#### How it works

Naive Bayes combines empirical class priors with smoothed token likelihoods. Use Laplace smoothing to avoid assigning zero probability to unseen terms. Compare with a majority baseline trained on the same partition. Its score puts any gain from text features into context.

##### Establish what the features add

Start with a majority classifier or a rule the existing workflow can explain. Fit it on the same training examples as the learned candidate and use the same test cases. A baseline should be cheap enough to maintain and realistic enough that beating it would matter.

A token-frequency model treats counts as evidence for a category. If the word “billing” occurs frequently in labeled payment tickets, it may raise that class's score for a new input. Smoothing prevents a word absent in one class from making the whole class impossible. A result may still depend too heavily on names or templates seen during training.

Compare errors on short and long requests. If the learned model improves only on long, well-formed examples, review whether that covers the actual intake. Check one false positive and one false negative with the desk team. Ask which available feature would correct the case and whether it would still exist at prediction time.

#### Worked example

A model trained on Wi-Fi and image-projector reports may learn the word projector. Spelling changes and overlapping words can still move its estimate unexpectedly.

#### Try it

Fit the two approaches on train. Print out predictions for validation and explain which words contribute to one mistake.


In [ ]:
import math
import re


def words(text):
    return re.findall(r"[a-z]+", text.lower())


def fit(records):
    labels = Counter(row.label for row in records)
    counts = {label: Counter() for label in labels}
    for row in records:
        counts[row.label].update(words(row.text))
    return {"labels": dict(labels), "counts": {label: dict(c) for label, c in counts.items()}}


def predict(model, text):
    labels, counts = model["labels"], model["counts"]
    vocabulary = {term for tokens in counts.values() for term in tokens}
    scores = {}
    for label, count in labels.items():
        likelihood_denominator = sum(counts[label].values()) + len(vocabulary)
        scores[label] = math.log(count / sum(labels.values())) + sum(
            math.log((counts[label].get(term, 0) + 1) / likelihood_denominator)
            for term in words(text) if term in vocabulary
        )
    best = max(sorted(scores), key=scores.get)
    confidence = 1 / sum(math.exp(score - scores[best]) for score in scores.values())
    return best, confidence


model = fit(splits["train"])
majority = max(sorted(model["labels"]), key=model["labels"].get)
for row in splits["validation"]:
    print(row.label, majority, predict(model, row.text))
assert sum(model["labels"].values()) == len(splits["train"])
assert predict(model, "projector screen")[0] == "display"


**Compare your result**

Keep priors and token counts tied to train. Explain the error rather than tuning directly on the test split.


## Choose a threshold and package evidence


### Lecture 03 · Measure selective risk and cost

Pick an abstention threshold on validation and report coverage and accuracy on test.

#### A better accepted score may cost more

A stricter threshold improves accuracy among answered cases and doubles the cases sent to the help desk.

Report accepted error, coverage, and review workload together. Choose the threshold on validation using an explicit error and handoff cost, then report results on held-out cases once.

#### How it works

A selective classifier can hand uncertain cases to a person. Accuracy among answered cases can look high if the system answers very little, so report coverage alongside it. Use independently estimated costs for mistakes and reviews. Choose the threshold with validation data and freeze it before evaluating test.

##### A decision threshold changes who does the work

A system can route high-confidence requests and send the rest to staff. Define **coverage** as the fraction of requests handled automatically and **selective risk** as the error rate on those accepted requests. Moving the threshold trades speed and queue volume against the number of incorrect automatic routes.

Imagine one setting accepts 80 of 100 requests and misroutes 8. Another accepts 60 and misroutes 3. Selective risk drops from 10% to 5%, while review volume doubles from 20 to 40. A recommendation needs operational costs as well as acceptance accuracy. Some wrong routes may be more costly than a manual review.

Select the threshold using validation data and a cost assumption appropriate to the workload. Report the final result on the held-out test, including uncertainty and per-group counts. Recheck that confidence estimates are usable on new traffic before assuming a threshold chosen from one set transfers to another.

#### Worked example

If one mistaken route costs four minutes and a review costs one, a threshold may be valuable even when accuracy changes little. A small sample can only illustrate the trade-off.

#### Try it

Compare the tested thresholds. Increase review cost and see which threshold you would choose on validation.


In [ ]:
from collections import defaultdict


def evaluate(records, predictor, threshold, review_cost=1, error_cost=4):
    summary = defaultdict(Counter)
    for row in records:
        predicted, confidence = predictor(row.text)
        outcome = "handoff" if confidence < threshold else ("correct" if predicted == row.label else "wrong")
        summary["all"][outcome] += 1
        summary[row.slice][outcome] += 1
    result = {}
    for key, counts in summary.items():
        accepted = counts["correct"] + counts["wrong"]
        result[key] = {
            "n": counts.total(), "wrong_accepted": counts["wrong"], "handoffs": counts["handoff"],
            "coverage": accepted / counts.total(),
            "accepted_accuracy": counts["correct"] / accepted if accepted else None,
            "cost": review_cost * counts["handoff"] + error_cost * counts["wrong"],
        }
    return result


def trained(text):
    return predict(model, text)


thresholds = (0.5, 0.65, 0.8, 0.9)
validation = {t: evaluate(splits["validation"], trained, t)["all"] for t in thresholds}
threshold = min(thresholds, key=lambda t: (validation[t]["cost"], -validation[t]["coverage"], t))
baseline = evaluate(splits["test"], lambda text: (majority, 1), 0)
report = evaluate(splits["test"], trained, threshold)
print("Validation", validation)
print("Frozen threshold", threshold, "baseline", baseline, "test", report)
assert report["all"]["n"] == len(splits["test"])
assert report["all"]["cost"] < baseline["all"]["cost"]


**Compare your result**

Report accepted count, wrong accepted count, and handoffs on each slice. Explain any change through its expected operating cost.


### Lecture 04 · Version a model artifact and regression gate

Serialize a fitted model, reload it, and gate release using both quality and coverage.

#### Save what the operator will actually run

The model artifact reloads correctly. Another team loads it with a different threshold because that setting was stored in a notebook cell.

Version parameters, preprocessing, threshold, and split evidence together. Run an artifact round trip and a gate check so the deployed decision matches the one that reviewers evaluated.

#### How it works

Persist parameters, label order, model configuration, and dataset lineage. Check loaded predictions against the in-memory result. Gate on meaningful evidence and inspect slice sample counts before trusting a pass. Changing parameters or the decision threshold should produce a new artifact and reevaluation.

##### A release includes the decision rule

Saving only model weights is insufficient if another environment uses different tokenization, a different threshold, or a different label order. Package the estimator with input normalization, output mapping, version identifiers, and the operating point that was evaluated.

Make a known-case round trip. Save the artifact, reload it through the same interface as serving, and compare predictions and fallback outcomes on a narrow fixture. Include a request at or close to the threshold. That case can reveal small version differences that a simple serialization check will miss.

Tie each release to a replayable evaluation record. Declare which failing slice would block deployment, and which previously approved artifact can be restored. After switching versions, verify the result through the serving route. An unchanged file hash confirms you loaded an object, while the round trip checks that people still receive the intended decision.

#### Worked example

A new model achieves better accepted accuracy by deferring half the traffic. A minimum coverage requirement surfaces the extra operational burden.

#### Try it

Round-trip the model. Tighten the coverage requirement and observe when the gate changes.


In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

artifact = {"dataset": dataset_version, "model_version": "nb-v1", "threshold": threshold, "parameters": model}
with TemporaryDirectory() as directory:
    artifact_path = Path(directory) / "classifier.json"
    artifact_path.write_text(json.dumps(artifact), encoding="utf-8")
    restored = json.loads(artifact_path.read_text(encoding="utf-8"))

for row in splits["test"]:
    assert predict(restored["parameters"], row.text) == trained(row.text)

overall = report["all"]
release = (
    overall["coverage"] >= 0.5
    and overall["accepted_accuracy"] is not None
    and overall["accepted_accuracy"] >= 0.8
    and overall["cost"] < baseline["all"]["cost"]
)
print("Release decision for this fixture", "pass" if release else "hold", restored["model_version"])


**Compare your result**

Round-trip results should match exactly. Record a keep, revise, or release decision with the evaluated version and the reason.


## Course project

Deliver an experiment report and a saved classifier artifact for routing synthetic service requests. Compare a majority baseline with a trained model, inspect missed cases by slice, and justify an operating threshold.

Demonstrate group-disjoint splits, fit the model only on train, pick the threshold on validation, and report the untouched test results. Include false positives, abstentions, an artifact round trip, and a regression gate.
